In [0]:
%sql

SHOW SCHEMAS in bootcamp

In [0]:
%sql

---Creating silver schema

CREATE SCHEMA IF NOT EXISTS bootcamp.silver 
COMMENT 'Capa Silver: Datos limpios, validados y con tipos correctos';

DESCRIBE SCHEMA bootcamp.silver


In [0]:
%sql

---Creating  gold Schema, this if for the final business outpu
CREATE SCHEMA IF NOT EXISTS bootcamp.gold 

COMMENT 'Capa Gold: Modelado para el negocio sobre el cuál realizar calculos de métricas';

DESCRIBE SCHEMA bootcamp.gold;


In [0]:
%sql
DROP TABLE IF EXISTS bootcamp.silver.propiedades;

CREATE TABLE bootcamp.silver.propiedades (
    propiedad_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT 'ID único de la propiedad (auto-generado)',

    partido STRING COMMENT 'Partido/municipio estandarizado',
    region STRING COMMENT 'Región geográfica: capital federal, gba zona norte/oeste/sur',

    tipo_operacion STRING COMMENT 'alquiler, venta, alquiler_temporario',
    precio DECIMAL(15,2) COMMENT 'Precio de la propiedad',
    moneda STRING COMMENT 'USD o ARS',
    expensas DECIMAL(15,2) COMMENT 'Expensas mensuales',

    ambientes INT COMMENT 'Cantidad de ambientes',
    metros_cuadrados_totales DECIMAL(15,2) COMMENT 'Superficie total',
    metros_cuadrados_cubiertos DECIMAL(15,2) COMMENT 'Superficie cubierta',

    antiguedad INT COMMENT 'Años de antigüedad (999 si no disponible)',
    cochera BOOLEAN COMMENT 'Tiene cochera',
    orientacion STRING COMMENT 'Orientación del inmueble',
    estado STRING COMMENT 'Estado de la propiedad',
    url STRING COMMENT 'URL de la propiedad',

    fecha_publicacion DATE COMMENT 'Fecha de publicación — desde fecha (STRING) en Bronze',

    precio_por_m2 DECIMAL(15,2) COMMENT 'Precio por metro cuadrado',

    _source_table STRING DEFAULT 'bootcamp.bronze.properties_bronze' COMMENT 'Tabla origen',
    _processing_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP() COMMENT 'Timestamp de procesamiento'
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Propiedades inmobiliarias - Capa Silver (datos limpios y validados)';


In [0]:
%sql

describe table extended bootcamp.silver.propiedades

In [0]:
%sql
-- propiedades_clean: casting básico de tipos (Semana 2)
CREATE OR REPLACE TEMPORARY VIEW propiedades_clean AS
SELECT
  CASE WHEN precio RLIKE '^[^a-zA-Z]+$' THEN precio::double ELSE NULL END AS precio,
  moneda,
  CASE WHEN ambientes RLIKE '^[^a-zA-Z]+$' THEN ambientes::double ELSE NULL END AS ambientes,
  CASE WHEN metros_cuadrados_totales RLIKE '^[^a-zA-Z]+$' THEN metros_cuadrados_totales::double ELSE NULL END AS metros_cuadrados_totales,
  CASE WHEN metros_cuadrados_cubiertos RLIKE '^[^a-zA-Z]+$' THEN metros_cuadrados_cubiertos::double ELSE NULL END AS metros_cuadrados_cubiertos,
  CASE WHEN antiguedad RLIKE '^[^a-zA-Z]+$' THEN antiguedad::double ELSE NULL END AS antiguedad,
  tipo_de_operacion, id, ubicacion, numero, calle, expensas,
  orientacion_cardinal, orientacion_inmueble, piso, cochera, estado,
  tipo_vendedor, url, zona, fecha, hora
FROM bootcamp.bronze.properties_bronze;

-- propiedades_clean_2: filtro de outliers por percentiles (Semana 2)
CREATE OR REPLACE TEMPORARY VIEW propiedades_clean_2 AS
(
  WITH limites AS (
    SELECT moneda, tipo_de_operacion,
      PERCENTILE(precio, 0.01) AS p01,
      PERCENTILE(precio, 0.99) AS p99
    FROM propiedades_clean
    WHERE precio > 0
      AND moneda IN ('USD', 'ARS')
      AND tipo_de_operacion IN ('venta', 'alquiler')
    GROUP BY moneda, tipo_de_operacion
  )
  SELECT p.*
  FROM propiedades_clean p
  JOIN limites l
    ON p.moneda = l.moneda
    AND p.tipo_de_operacion = l.tipo_de_operacion
  WHERE p.precio BETWEEN l.p01 AND l.p99
);

-- bronze_EDA: limpieza detallada (Semana 2)
CREATE OR REPLACE TEMP VIEW bronze_EDA AS (
  SELECT
    edl.id,
    edl.ubicacion,
    CASE
      WHEN edl.precio = 'NaN' OR edl.precio NOT RLIKE '^-?[0-9]+\\.?[0-9]*$' THEN NULL
      WHEN edl.precio::float BETWEEN 0 AND 2147483648 THEN edl.precio::float
      ELSE NULL
    END AS precio,
    CASE
      WHEN edl.numero = 'NaN' OR edl.numero NOT RLIKE '^-?[0-9]+\\.?[0-9]*$' THEN NULL
      WHEN edl.numero::float BETWEEN -10000 AND 50000 THEN edl.numero::float
      ELSE NULL
    END AS numero,
    edl.calle,
    CASE
      WHEN edl.expensas = 'NaN' OR edl.expensas NOT RLIKE '^-?[0-9]+\\.?[0-9]*$' THEN NULL
      WHEN edl.expensas::float BETWEEN 0 AND 20000000 THEN edl.expensas::float
      ELSE NULL
    END AS expensas,
    edl.tipo_de_operacion,
    CASE
      WHEN lower(edl.moneda) LIKE '%dolares%' THEN 'USD'
      WHEN lower(edl.moneda) LIKE '%us%' THEN 'USD'
      WHEN lower(edl.moneda) LIKE '%pesos%' THEN 'ARS'
      WHEN lower(edl.moneda) LIKE '%ars%' THEN 'ARS'
      ELSE edl.moneda
    END AS moneda,
    CASE
      WHEN edl.ambientes = 'NaN' OR edl.ambientes NOT RLIKE '^-?[0-9]+\\.?[0-9]*$' THEN NULL
      ELSE edl.ambientes::float
    END AS ambientes,
    CASE
      WHEN edl.metros_cuadrados_totales = 'NaN' OR edl.metros_cuadrados_totales NOT RLIKE '^-?[0-9]+\\.?[0-9]*$' THEN NULL
      ELSE edl.metros_cuadrados_totales::decimal
    END AS metros_cuadrados_totales,
    CASE
      WHEN edl.metros_cuadrados_cubiertos = 'NaN' OR edl.metros_cuadrados_cubiertos NOT RLIKE '^-?[0-9]+\\.?[0-9]*$' THEN NULL
      ELSE edl.metros_cuadrados_cubiertos::decimal
    END AS metros_cuadrados_cubiertos,
    edl.orientacion_cardinal,
    edl.orientacion_inmueble,
    CASE
      WHEN edl.piso IS NULL THEN NULL
      WHEN edl.piso = 'NaN' THEN NULL
      ELSE edl.piso::float
    END AS piso,
    CASE
      WHEN edl.cochera = 'tiene' THEN 1
      ELSE NULL
    END AS cochera,
    edl.antiguedad,
    edl.estado,
    edl.tipo_vendedor,
    edl.url,
    edl.zona,
    COALESCE(CAST(edl.fecha AS DATE), CURRENT_DATE()) AS fecha
  FROM propiedades_clean_2 edl
  WHERE edl.url RLIKE 'https'
    AND (edl.zona LIKE '%gba%' OR edl.zona LIKE '%caba%' OR edl.zona LIKE '%capital%')
    AND edl.zona LIKE '%-%'
    AND LENGTH(edl.zona) < 50
);

SELECT COUNT(*) as registros_bronze_eda FROM bronze_EDA;


In [0]:
%sql
WITH datos_tipados AS (
    SELECT
        LOWER(TRIM(zona)) as zona,
        LOWER(TRIM(tipo_de_operacion)) as tipo_operacion,
        CAST(precio AS DECIMAL(15,2)) as precio,
        moneda,
        COALESCE(CAST(expensas AS DECIMAL(15,2)), 0) as expensas,
        CAST(ambientes AS INT) as ambientes,
        CAST(metros_cuadrados_totales AS DECIMAL(15,2)) as metros_cuadrados_totales,
        CAST(metros_cuadrados_cubiertos AS DECIMAL(15,2)) as metros_cuadrados_cubiertos,
        CASE
            WHEN antiguedad RLIKE '^[0-9]' AND CAST(antiguedad AS FLOAT) BETWEEN 0 AND 200
                 THEN CAST(antiguedad AS INT)
            ELSE 999
        END as antiguedad,
        CASE WHEN cochera = 1 THEN TRUE ELSE FALSE END as cochera,
        COALESCE(TRIM(orientacion_inmueble), '-1') as orientacion,
        COALESCE(TRIM(estado), '-1') as estado,
        url,
        fecha as fecha_publicacion
    FROM bronze_EDA
    WHERE precio > 0
      AND metros_cuadrados_totales > 0
      AND tipo_de_operacion IS NOT NULL
)
SELECT *, typeof(precio) as tipo_precio, typeof(ambientes) as tipo_ambientes
FROM datos_tipados
LIMIT 10;

In [0]:
%sql
WITH datos_tipados AS (
    SELECT
        LOWER(TRIM(zona)) as zona,
        LOWER(TRIM(tipo_de_operacion)) as tipo_operacion,
        CAST(precio AS DECIMAL(15,2)) as precio,
        moneda,
        COALESCE(CAST(expensas AS DECIMAL(15,2)), 0) as expensas,
        CAST(ambientes AS INT) as ambientes,
        CAST(metros_cuadrados_totales AS DECIMAL(15,2)) as metros_cuadrados_totales,
        CAST(metros_cuadrados_cubiertos AS DECIMAL(15,2)) as metros_cuadrados_cubiertos,
        CASE
            WHEN antiguedad RLIKE '^[0-9]' AND CAST(antiguedad AS FLOAT) BETWEEN 0 AND 200
                 THEN CAST(antiguedad AS INT)
            ELSE 999
        END as antiguedad,
        CASE WHEN cochera = 1 THEN TRUE ELSE FALSE END as cochera,
        COALESCE(TRIM(orientacion_inmueble), '-1') as orientacion,
        COALESCE(TRIM(estado), '-1') as estado,
        url,
        fecha as fecha_publicacion
    FROM bronze_EDA
    WHERE precio > 0
      AND metros_cuadrados_totales > 0
      AND tipo_de_operacion IS NOT NULL
),
datos_deduplicados AS (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY precio, url ORDER BY precio) as rn
    FROM datos_tipados
)
SELECT
    COUNT(*) as total,
    COUNT(CASE WHEN rn = 1 THEN 1 END) as unicos,
    COUNT(CASE WHEN rn > 1 THEN 1 END) as duplicados
FROM datos_deduplicados;

In [0]:
%sql
INSERT INTO bootcamp.silver.propiedades (
    partido,
    region,
    tipo_operacion,
    precio,
    moneda,
    expensas,
    ambientes,
    metros_cuadrados_totales,
    metros_cuadrados_cubiertos,
    antiguedad,
    cochera,
    orientacion,
    estado,
    url,
    fecha_publicacion,
    precio_por_m2,
    _source_table,
    _processing_timestamp
)

WITH
-- CTE 1: Tipos finales desde bronze_EDA
datos_tipados AS (
    SELECT
        LOWER(TRIM(zona)) as zona,
        LOWER(TRIM(tipo_de_operacion)) as tipo_operacion,
        CAST(precio AS DECIMAL(15,2)) as precio,
        moneda,
        COALESCE(CAST(expensas AS DECIMAL(15,2)), 0) as expensas,
        CAST(ambientes AS INT) as ambientes,
        CAST(metros_cuadrados_totales AS DECIMAL(15,2)) as metros_cuadrados_totales,
        CAST(metros_cuadrados_cubiertos AS DECIMAL(15,2)) as metros_cuadrados_cubiertos,
        CASE
            WHEN antiguedad RLIKE '^[0-9]' AND CAST(antiguedad AS FLOAT) BETWEEN 0 AND 200
                 THEN CAST(antiguedad AS INT)
            ELSE 999
        END as antiguedad,
        CASE WHEN cochera = 1 THEN TRUE ELSE FALSE END as cochera,
        COALESCE(TRIM(orientacion_inmueble), '-1') as orientacion,
        COALESCE(TRIM(estado), '-1') as estado,
        url,
        fecha as fecha_publicacion
    FROM bronze_EDA
    WHERE precio > 0
      AND metros_cuadrados_totales > 0
      AND tipo_de_operacion IS NOT NULL
),

-- CTE 2: Deduplicación
datos_deduplicados AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY precio, url
            ORDER BY precio
        ) as rn
    FROM datos_tipados
),

-- CTE 3: partido + region + precio_por_m2
datos_finales AS (
    SELECT
        CASE
            WHEN zona = 'capital-federal' THEN 'capital federal'
            WHEN zona = 'bsas-gba-norte/vicente-lopez' THEN 'vicente lopez'
            WHEN zona = 'bsas-gba-norte/pilar' THEN 'pilar'
            WHEN zona = 'bsas-gba-norte/tigre' THEN 'tigre'
            WHEN zona = 'bsas-gba-norte/san-isidro' THEN 'san isidro'
            WHEN zona = 'bsas-gba-norte/general-san-martin' THEN 'general san martin'
            WHEN zona = 'bsas-gba-norte/san-fernando' THEN 'san fernando'
            WHEN zona = 'bsas-gba-norte/san-miguel' THEN 'san miguel'
            WHEN zona = 'bsas-gba-norte/malvinas-argentinas' THEN 'malvinas argentinas'
            WHEN zona = 'bsas-gba-norte/escobar' THEN 'escobar'
            WHEN zona = 'bsas-gba-oeste/caseros' THEN 'tres de febrero'
            WHEN zona = 'bsas-gba-oeste/castelar' THEN 'moron'
            WHEN zona = 'bsas-gba-oeste/general-rodriguez' THEN 'general rodriguez'
            WHEN zona = 'bsas-gba-oeste/hurlingham' THEN 'hurlingham'
            WHEN zona = 'bsas-gba-oeste/ituzaingo' THEN 'ituzaingo'
            WHEN zona = 'bsas-gba-oeste/la-matanza' THEN 'la matanza'
            WHEN zona = 'bsas-gba-oeste/merlo' THEN 'merlo'
            WHEN zona = 'bsas-gba-oeste/moreno' THEN 'moreno'
            WHEN zona = 'bsas-gba-oeste/moron' THEN 'moron'
            WHEN zona = 'bsas-gba-oeste/tres-de-febrero' THEN 'tres de febrero'
            WHEN zona = 'bsas-gba-sur/almirante-brown' THEN 'almirante brown'
            WHEN zona = 'bsas-gba-sur/avellaneda' THEN 'avellaneda'
            WHEN zona = 'bsas-gba-sur/berazategui' THEN 'berazategui'
            WHEN zona = 'bsas-gba-sur/esteban-echeverria' THEN 'esteban echeverria'
            WHEN zona = 'bsas-gba-sur/ezeiza' THEN 'ezeiza'
            WHEN zona = 'bsas-gba-sur/florencio-varela' THEN 'florencio varela'
            WHEN zona = 'bsas-gba-sur/lanus' THEN 'lanus'
            WHEN zona = 'bsas-gba-sur/la-plata' THEN 'la plata'
            WHEN zona = 'bsas-gba-sur/lomas-de-zamora' THEN 'lomas de zamora'
            WHEN zona = 'bsas-gba-sur/quilmes' THEN 'quilmes'
            WHEN zona RLIKE 'vicente-lopez' THEN 'vicente lopez'
            WHEN zona RLIKE 'pilar' THEN 'pilar'
            WHEN zona RLIKE 'general-san-martin' THEN 'general san martin'
            WHEN zona RLIKE 'tigre' THEN 'tigre'
            WHEN zona RLIKE 'san-fernando' THEN 'san fernando'
            WHEN zona RLIKE 'san-miguel' THEN 'san miguel'
            WHEN zona RLIKE 'malvinas-argentinas' THEN 'malvinas argentinas'
            WHEN zona RLIKE 'escobar' THEN 'escobar'
            WHEN zona RLIKE 'caseros' THEN 'tres de febrero'
            WHEN zona RLIKE 'castelar' THEN 'moron'
            WHEN zona RLIKE 'general-rodriguez' THEN 'general rodriguez'
            WHEN zona RLIKE 'hurlingham' THEN 'hurlingham'
            WHEN zona RLIKE 'ituzaingo' THEN 'ituzaingo'
            WHEN zona RLIKE 'la-matanza' THEN 'la matanza'
            WHEN zona RLIKE 'merlo' THEN 'merlo'
            WHEN zona RLIKE 'moreno' THEN 'moreno'
            WHEN zona RLIKE 'moron' THEN 'moron'
            WHEN zona RLIKE 'tres-de-febrero' THEN 'tres de febrero'
            WHEN zona RLIKE 'almirante-brown' THEN 'almirante brown'
            WHEN zona RLIKE 'avellaneda' THEN 'avellaneda'
            WHEN zona RLIKE 'berazategui' THEN 'berazategui'
            WHEN zona RLIKE 'berisso' THEN 'berisso'
            WHEN zona RLIKE 'ensenada' THEN 'ensenada'
            WHEN zona RLIKE 'esteban-echeverria' THEN 'esteban echeverria'
            WHEN zona RLIKE 'ezeiza' THEN 'ezeiza'
            WHEN zona RLIKE 'florencio-varela' THEN 'florencio varela'
            WHEN zona RLIKE 'lanus' THEN 'lanus'
            WHEN zona RLIKE 'la-plata' THEN 'la plata'
            WHEN zona RLIKE 'lomas-de-zamora' THEN 'lomas de zamora'
            WHEN zona RLIKE 'presidente-peron' THEN 'presidente peron'
            WHEN zona RLIKE 'quilmes' THEN 'quilmes'
            WHEN zona RLIKE 'san-vicente' THEN 'san vicente'
            WHEN zona = 'san-isidro' THEN 'san isidro'
            WHEN zona = 'gba-zona-norte--vicente-lopez' THEN 'vicente lopez'
            WHEN zona = 'gba-zona-norte--pilar' THEN 'pilar'
            WHEN zona = 'gba-zona-norte--general-san-martin' THEN 'general san martin'
            WHEN zona = 'gba-zona-norte--san-isidro' THEN 'san isidro'
            WHEN zona = 'gba-zona-norte--tigre' THEN 'tigre'
            WHEN zona = 'gba-zona-norte--san-fernando' THEN 'san fernando'
            WHEN zona = 'gba-zona-norte--san-miguel' THEN 'san miguel'
            WHEN zona = 'gba-zona-norte--malvinas-argentinas' THEN 'malvinas argentinas'
            WHEN zona = 'gba-zona-norte--escobar' THEN 'escobar'
            WHEN zona = 'gba-zona-oeste--canuelas' THEN 'canuelas'
            WHEN zona = 'gba-zona-oeste--general-rodriguez' THEN 'general rodriguez'
            WHEN zona = 'gba-zona-oeste--hurlingham' THEN 'hurlingham'
            WHEN zona = 'gba-zona-oeste--ituzaingo' THEN 'ituzaingo'
            WHEN zona = 'gba-zona-oeste--la-matanza' THEN 'la matanza'
            WHEN zona = 'gba-zona-oeste--lujan' THEN 'lujan'
            WHEN zona = 'gba-zona-oeste--marcos-paz' THEN 'marcos paz'
            WHEN zona = 'gba-zona-oeste--merlo' THEN 'merlo'
            WHEN zona = 'gba-zona-oeste--moreno' THEN 'moreno'
            WHEN zona = 'gba-zona-oeste--moron' THEN 'moron'
            WHEN zona = 'gba-zona-oeste--tres-de-febrero' THEN 'tres de febrero'
            WHEN zona = 'gba-zona-sur--almirante-brown' THEN 'almirante brown'
            WHEN zona = 'gba-zona-sur--avellaneda' THEN 'avellaneda'
            WHEN zona = 'gba-zona-sur--berazategui' THEN 'berazategui'
            WHEN zona = 'gba-zona-sur--ensenada' THEN 'ensenada'
            WHEN zona = 'gba-zona-sur--san-vicente' THEN 'san vicente'
            WHEN zona = 'gba-zona-sur--esteban-echeverria' THEN 'esteban echeverria'
            WHEN zona = 'gba-zona-sur--ezeiza' THEN 'ezeiza'
            WHEN zona = 'gba-zona-sur--florencio-varela' THEN 'florencio varela'
            WHEN zona = 'gba-zona-sur--lanus' THEN 'lanus'
            WHEN zona = 'gba-zona-sur--la-plata' THEN 'la plata'
            WHEN zona = 'gba-zona-sur--lomas-de-zamora' THEN 'lomas de zamora'
            WHEN zona = 'gba-zona-sur--presidente-peron' THEN 'presidente peron'
            WHEN zona = 'gba-zona-sur--quilmes' THEN 'quilmes'
            ELSE 'no especifica'
        END as partido,

        CASE
            WHEN zona IS NULL THEN 'no especifica'
            WHEN zona = 'capital-federal' THEN 'capital federal'
            WHEN zona LIKE 'bsas-gba-norte/%' THEN 'gba zona norte'
            WHEN zona IN ('vicente-lopez','pilar','tigre','san-isidro','general-san-martin',
                         'san-fernando','san-miguel','jose-c-paz','malvinas-argentinas','escobar') THEN 'gba zona norte'
            WHEN zona LIKE 'bsas-gba-oeste/%' THEN 'gba zona oeste'
            WHEN zona IN ('caseros','castelar','general-rodriguez','hurlingham','ituzaingo',
                         'la-matanza','merlo','moreno','moron','tres-de-febrero') THEN 'gba zona oeste'
            WHEN zona LIKE 'bsas-gba-sur/%' THEN 'gba zona sur'
            WHEN zona IN ('almirante-brown','avellaneda','berazategui','berisso','ensenada',
                         'esteban-echeverria','ezeiza','florencio-varela','lanus','la-plata',
                         'lomas-de-zamora','presidente-peron','quilmes','san-vicente') THEN 'gba zona sur'
            WHEN zona LIKE 'gba-zona-norte--%' THEN 'gba zona norte'
            WHEN zona LIKE 'gba-zona-oeste--%' THEN 'gba zona oeste'
            WHEN zona LIKE 'gba-zona-sur--%' THEN 'gba zona sur'
            ELSE zona
        END as region,

        tipo_operacion, precio, moneda, expensas, ambientes,
        metros_cuadrados_totales, metros_cuadrados_cubiertos,
        antiguedad, cochera, orientacion, estado, url, fecha_publicacion,
        ROUND(precio / metros_cuadrados_totales, 2) as precio_por_m2
    FROM datos_deduplicados
    WHERE rn = 1
)

SELECT
    partido, region, tipo_operacion,
    precio, moneda, expensas, ambientes,
    metros_cuadrados_totales, metros_cuadrados_cubiertos,
    antiguedad, cochera, orientacion, estado, url, fecha_publicacion,
    precio_por_m2,
    'bootcamp.bronze.properties_bronze' as _source_table,
    CURRENT_TIMESTAMP() as _processing_timestamp
FROM datos_finales;


In [0]:
%sql
SELECT
    'Bronze (properties_bronze)' as capa,
    COUNT(*) as registros
FROM bootcamp.bronze.properties_bronze

UNION ALL

SELECT
    'Silver (propiedades)' as capa,
    COUNT(*) as registros
FROM bootcamp.silver.propiedades;


In [0]:
%sql
SELECT
    COUNT(*) as total_silver,
    COUNT(CASE WHEN precio <= 0 THEN 1 END) as precio_invalido,
    COUNT(CASE WHEN metros_cuadrados_totales <= 0 THEN 1 END) as m2_invalido,
    COUNT(CASE WHEN antiguedad = 999 THEN 1 END) as antiguedad_placeholder_999,
    COUNT(CASE WHEN antiguedad BETWEEN 0 AND 200 THEN 1 END) as antiguedad_valida,
    COUNT(CASE WHEN partido IS NULL OR partido = '' THEN 1 END) as partido_vacio,
    COUNT(CASE WHEN precio_por_m2 IS NULL THEN 1 END) as precio_m2_nulo,
    COUNT(CASE WHEN precio_por_m2 > 0 THEN 1 END) as precio_m2_valido
FROM bootcamp.silver.propiedades;


In [0]:
%sql
SELECT
    partido,
    tipo_operacion,
    moneda,
    COUNT(*) as total_propiedades,
    ROUND(AVG(precio), 2) as precio_promedio,
    ROUND(PERCENTILE(precio, 0.5), 2) as precio_mediana,
    ROUND(MIN(precio), 2) as precio_minimo,
    ROUND(MAX(precio), 2) as precio_maximo,
    ROUND(AVG(precio_por_m2), 2) as precio_m2_promedio,
    ROUND(AVG(metros_cuadrados_totales), 2) as m2_promedio,
    ROUND(AVG(ambientes), 1) as ambientes_promedio
FROM bootcamp.silver.propiedades
GROUP BY partido, tipo_operacion, moneda
HAVING COUNT(*) >= 5
ORDER BY precio_promedio DESC
LIMIT 15;


In [0]:
%sql
WITH precios_partido AS (
    SELECT
        partido,
        tipo_operacion,
        moneda,
        COUNT(*) as total_propiedades,
        ROUND(AVG(precio), 2) as precio_promedio,
        ROUND(AVG(precio_por_m2), 2) as precio_m2_promedio
    FROM bootcamp.silver.propiedades
    GROUP BY partido, tipo_operacion, moneda
    HAVING COUNT(*) >= 10
)
SELECT
    partido,
    tipo_operacion,
    moneda,
    total_propiedades,
    precio_promedio,
    precio_m2_promedio,
    ROW_NUMBER() OVER (
        PARTITION BY tipo_operacion, moneda
        ORDER BY precio_promedio DESC
    ) as ranking
FROM precios_partido
ORDER BY tipo_operacion, moneda, ranking
LIMIT 20;


In [0]:
%sql
SELECT
    ambientes,
    tipo_operacion,
    moneda,
    COUNT(*) as total_propiedades,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY tipo_operacion, moneda), 2) as pct_del_total,
    ROUND(AVG(precio), 2) as precio_promedio,
    ROUND(AVG(metros_cuadrados_totales), 2) as m2_promedio
FROM bootcamp.silver.propiedades
WHERE ambientes IS NOT NULL AND ambientes > 0
GROUP BY ambientes, tipo_operacion, moneda
ORDER BY tipo_operacion, moneda, ambientes;


In [0]:
%sql
SELECT 'BRONZE' as capa, table_name, table_type
FROM bootcamp.information_schema.tables
WHERE table_schema = 'bronze'

UNION ALL

SELECT 'SILVER' as capa, table_name, table_type
FROM bootcamp.information_schema.tables
WHERE table_schema = 'silver'

UNION ALL

SELECT 'GOLD' as capa, table_name, table_type
FROM bootcamp.information_schema.tables
WHERE table_schema = 'gold'

ORDER BY capa, table_name;


In [0]:
%sql
SELECT 'Bronze (properties_bronze)' as tabla, COUNT(*) as registros
FROM bootcamp.bronze.properties_bronze

UNION ALL

SELECT 'Silver (propiedades)' as tabla, COUNT(*) as registros
FROM bootcamp.silver.propiedades;


In [0]:
%sql
SELECT
    'precio <= 0' as motivo_filtro,
    COUNT(*) as registros_excluidos
FROM bootcamp.bronze.properties_bronze
WHERE CAST(precio AS DOUBLE) <= 0 OR precio IS NULL

UNION ALL

SELECT
    'm2 <= 0' as motivo_filtro,
    COUNT(*) as registros_excluidos
FROM bootcamp.bronze.properties_bronze
WHERE CAST(metros_cuadrados_totales AS DOUBLE) <= 0 OR metros_cuadrados_totales IS NULL

UNION ALL

SELECT
    'tipo_operacion NULL' as motivo_filtro,
    COUNT(*) as registros_excluidos
FROM bootcamp.bronze.properties_bronze
WHERE tipo_de_operacion IS NULL

UNION ALL

SELECT
    'zona no mapeada' as motivo_filtro,
    COUNT(*) as registros_excluidos
FROM bootcamp.bronze.properties_bronze
WHERE zona IS NULL OR zona = ''

UNION ALL

SELECT
    'TOTAL Bronze' as motivo_filtro,
    COUNT(*) as registros_excluidos
FROM bootcamp.bronze.properties_bronze;
